# SigAlg's `ProbabilityMeasure` class

In [ ]:
# If running in Google Colab, uncomment the line below and run this cell first.
# Also, for Mac+Chrome users, beware of a known bug with LaTeX redering in Colab: https://github.com/googlecolab/colabtools/issues/3192

# !pip install sigalg

The `ProbabilityMeasure` class in SigAlg is the fundamental class for representing probability measures on sample spaces. The API reference is [here](https://johnmyers-phd.com/sigalg/api/modules/core/#sigalg.core.ProbabilityMeasure).

## Mathematical definition

Let $(\Omega, \mathcal{F})$ be a measurable space consisting of a $\sigma$-algebra $\mathcal{F}$ on a set $\Omega$. A *probability measure* $P$ is a countably additive function $P: \mathcal{F} \to [0,1]$ such that $P(\Omega) = 1$. Here, *countable additivity* means that

$$
P \left( \bigcup_{k=1}^\infty A_k \right) = \sum_{k=1}^\infty P(A_k)
$$

for all collections $\{A_k\}_{k=1}^\infty$ of pairwise disjoint measurable sets. If $\Omega$ is finite (as it always is, in SigAlg), then $P$ needs only to be finitely additive in order to be countably additive.

Though according to this definition a probability measure is only defined on sets in a fixed $\sigma$-algebra, this requirement is not enforced in SigAlg. In particular, every probability measure $P$ in SigAlg is defined on the power-set $\sigma$-algebra, meaning that we can evaluate $P$ at sample points:

$$
P(\omega) \stackrel{\mathrm{def}}{=} P(\{\omega\}),
$$

for each $\omega \in \Omega$. In this manner, the probability measure functions as a probability mass function.

In SigAlg, an instance `P` of `ProbabilityMeasure` represents such a probability measure. The instance carries:
- A `sample_space` attribute representing $\Omega$
- A `data` attribute (a `pd.Series`) representing the probability values
- A `probabilities` attribute (a dictionary) mapping sample points to their probabilities

## API examples

### Creating probability measures

#### From dictionaries

Begin by defining a sample space $\Omega = \{0,1,2,3,4\}$.

In [2]:
from sigalg.core import ProbabilityMeasure, SampleSpace

Omega = SampleSpace().from_sequence(size=5)

print(Omega)

Sample space 'Omega':
[0, 1, 2, 3, 4]


Create a probability measure $P$ from a dictionary of probabilities:

In [3]:
P = ProbabilityMeasure(sample_space=Omega, name="P").from_dict(
    {
        0: 0.1,
        1: 0.2,
        2: 0.3,
        3: 0.25,
        4: 0.15,
    }
)
print(P)

Probability measure 'P':
        probability
sample             
0              0.10
1              0.20
2              0.30
3              0.25
4              0.15


If the sample space is not provided, it will be automatically generated from the dictionary keys:

In [4]:
Q = ProbabilityMeasure(name="Q").from_dict(
    {
        "a": 0.4,
        "b": 0.35,
        "c": 0.25,
    }
)
print(Q, "\n")
print(Q.sample_space)

Probability measure 'Q':
        probability
sample             
a              0.40
b              0.35
c              0.25 

Sample space 'Omega':
['a', 'b', 'c']


#### From `pd.Series` objects

Create a probability measure from a series:

In [5]:
import pandas as pd

s = pd.Series([0.2, 0.5, 0.3], index=["x", "y", "z"])

R = ProbabilityMeasure(name="R").from_pandas(s)
print(R)

Probability measure 'R':
        probability
sample             
x               0.2
y               0.5
z               0.3


#### Random probability measures

Generate a random probability measure by sampling from a Dirichlet distribution:

In [6]:
import numpy as np

rng = np.random.default_rng(42)
Omega = SampleSpace().from_sequence(size=4)

P_random = ProbabilityMeasure(sample_space=Omega, name="P_random").from_rand(
    random_state=rng
)
print(P_random)

Probability measure 'P_random':
        probability
sample             
0          0.324676
1          0.315490
2          0.322049
3          0.037785


### Factory methods

#### The uniform distribution

The uniform probability measure assigns equal probability to all sample points:

In [7]:
Omega = SampleSpace().from_sequence(size=5)

P_uniform = ProbabilityMeasure.uniform(sample_space=Omega, name="P_uniform")
print(P_uniform)

Probability measure 'P_uniform':
        probability
sample             
0               0.2
1               0.2
2               0.2
3               0.2
4               0.2


#### From features of a random vector

Create a probability measure from a random vector and a probability mass function on its range:

In [8]:
from numbers import Real

from sigalg.core import FeatureVector, RandomVector

Omega = SampleSpace().from_sequence(size=4)

X = RandomVector(domain=Omega).from_dict(
    {
        0: (0, 0),
        1: (0, 1),
        2: (1, 0),
        3: (1, 1),
    }
)


def pmf(v: FeatureVector) -> Real:
    """Probability mass function on the range of X."""
    v0, v1 = v
    return 0.75**v0 * 0.25 ** (1 - v0) * 0.6**v1 * 0.4 ** (1 - v1)


P = ProbabilityMeasure.from_features(rv=X, pmf=pmf)
print(P)

Probability measure 'P':
        probability
sample             
0              0.10
1              0.15
2              0.30
3              0.45


### Properties of probability measures

#### Data and probabilities

Access the underlying data and probability mapping:

In [9]:
Omega = SampleSpace().from_sequence(size=3)
P = ProbabilityMeasure(sample_space=Omega, name="P").from_dict(
    {
        0: 0.2,
        1: 0.5,
        2: 0.3,
    }
)

print(f"Data:\n{P.data}\n")
print(f"Probabilities dictionary:\n{P.probabilities}\n")
print(P.sample_space)

Data:
sample
0    0.2
1    0.5
2    0.3
Name: probability, dtype: float64

Probabilities dictionary:
{0: 0.2, 1: 0.5, 2: 0.3}

Sample space 'Omega':
[0, 1, 2]


### Calling probability measures

#### Evaluating at a single sample point

Evaluate the probability measure at a sample point:

In [10]:
print(f"P(0) = {P(0)}")
print(f"P(1) = {P(1)}")
print(f"P(2) = {P(2)}")

P(0) = 0.2
P(1) = 0.5
P(2) = 0.3


#### Evaluating at a list of sample points

Compute the probability of an event given as a list of sample points:

In [11]:
print(f"P([0, 1]) = {P([0, 1])}")
print(f"P([1, 2]) = {P([1, 2])}")

P([0, 1]) = 0.7
P([1, 2]) = 0.8


#### Evaluating at an event

Compute the probability of an event given as an `Event` object:

In [12]:
A = Omega.get_event([0, 2], name="A")
B = Omega.get_event([1], name="B")

print(f"P(A) = {P(A)}")
print(f"P(B) = {P(B)}")

P(A) = 0.5
P(B) = 0.5


### Methods

#### Conditional probability

Given two events $A$ and $B$ with $P(B) > 0$, the *conditional probability* of $A$ given $B$ is

$$
P(A\mid B) = \frac{P(A \cap B)}{P(B)}.
$$

In [13]:
Omega = SampleSpace().from_sequence(size=5)
P = ProbabilityMeasure(sample_space=Omega, name="P").from_dict(
    {
        0: 0.1,
        1: 0.2,
        2: 0.3,
        3: 0.25,
        4: 0.15,
    }
)

A = Omega.get_event([0, 1, 2], name="A")
B = Omega.get_event([1, 2, 3], name="B")

cond_prob = P.conditional_probability(event=A, given=B)
print(f"P(A|B) = {cond_prob}\n")

# Verify the formula
print(f"P(A ∩ B) = {P(A & B)}")
print(f"P(B) = {P(B)}")
print(f"P(A ∩ B) / P(B) = {P(A & B) / P(B)}")

P(A|B) = 0.6666666666666666

P(A ∩ B) = 0.5
P(B) = 0.75
P(A ∩ B) / P(B) = 0.6666666666666666


#### Independence

Two events $A$ and $B$ are *independent* if $P(A \cap B) = P(A) P(B)$.

##### Independence of events

Check if two events are independent:

In [14]:
from scipy.stats import bernoulli

from sigalg.core import Time
from sigalg.processes import IIDProcess

# Flip a biased coin twice
time = Time.discrete(length=1)
coin_flips = IIDProcess(
    distribution=bernoulli(p=0.6),
    support=[0, 1],
    name="coin_flips",
    time=time,
).from_enumeration()

Omega = coin_flips.domain
P = coin_flips.probability_measure

print("Coin flips process:")
print(coin_flips, "\n")
print(P, "\n")

# Define events
first_heads = Omega.get_event([2, 3], name="first_heads")
second_heads = Omega.get_event([1, 3], name="second_heads")

print(
    f"Are 'first_heads' and 'second_heads' independent? {P.are_independent(event1=first_heads, event2=second_heads)}"
)

Coin flips process:
Stochastic process 'coin_flips':
time        0  1
trajectory      
0           0  0
1           0  1
2           1  0
3           1  1 

Probability measure 'P':
            probability
trajectory             
0                  0.16
1                  0.24
2                  0.24
3                  0.36 

Are 'first_heads' and 'second_heads' independent? True


##### Independence of random vectors

Two random vectors $X$ and $Y$ are *independent* if the $\sigma$-algebras they generate are independent.

In [15]:
# Define random variables for the outcomes of the coin flips
flip1, flip2 = coin_flips

print(f"Are flip1 and flip2 independent? {P.are_independent(rv1=flip1, rv2=flip2)}\n")

# Create a dependent random variable
sum_flips = flip1 + flip2
print(f"Are flip1 and sum_flips independent? {P.are_independent(rv1=flip1, rv2=sum_flips)}")

Are flip1 and flip2 independent? True

Are flip1 and sum_flips independent? False


#### Almost sure equality

Two random vectors $X$ and $Y$ are *equal almost surely* if

$$
P \left( \{\omega \in \Omega : X(\omega) \neq Y(\omega)\} \right) = 0.
$$

In [16]:
from sigalg.core import RandomVariable

Omega = SampleSpace().from_sequence(size=4)
P = ProbabilityMeasure(sample_space=Omega, name="P").from_dict(
    {
        0: 0.4,
        1: 0.3,
        2: 0.3,
        3: 0.0,  # Zero probability
    }
)

X = RandomVariable(domain=Omega, name="X").from_dict(
    {
        0: 1,
        1: 2,
        2: 3,
        3: 4,
    }
)

Y = RandomVariable(domain=Omega, name="Y").from_dict(
    {
        0: 1,
        1: 2,
        2: 3,
        3: 100,  # Different from X, but on zero-probability event
    }
)

Z = RandomVariable(domain=Omega, name="Z").from_dict(
    {
        0: 1,
        1: 2,
        2: 10,  # Different from X on positive-probability event
        3: 4,
    }
)

print(f"P(X = Y almost surely)? {P.almost_surely_equal(X, Y)}")
print(f"P(X = Z almost surely)? {P.almost_surely_equal(X, Z)}")

P(X = Y almost surely)? True
P(X = Z almost surely)? False
